In [1]:
!pip install -q groq fastapi celery redis supabase python-dotenv pydub soundfile numpy torch
!pip install -q git+https://github.com/OpenBMB/VoxCPM.git
!pip install -q marker-pdf
!apt-get install -q ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 28.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 12.8

In [2]:
%%writefile worker.py
import os
import time
import json
import subprocess
import gc
import shutil
import torch
import numpy as np
import soundfile as sf
from pydub import AudioSegment
from celery import Celery
from dotenv import load_dotenv
from supabase import create_client, Client
from groq import Groq
from voxcpm import VoxCPM

load_dotenv()

# Initialize Supabase
supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
bucket_name = os.getenv("SUPABASE_BUCKET", "audiobooks")

# Initialize Celery
redis_url = os.getenv("CELERY_BROKER_URL")
app = Celery("audiobook_tasks", broker=redis_url, backend=redis_url)
app.conf.update(
    task_acks_late=True,
    task_reject_on_worker_lost=True,
    broker_use_ssl={"ssl_cert_reqs": "none"},
    redis_backend_use_ssl={"ssl_cert_reqs": "none"},
)

@app.task(name="worker.process_audiobook", bind=True)
def process_audiobook(self, task_id, pdf_storage_path, mp3_storage_path):
    start_time = time.time()
    telemetry = {"task_id": task_id, "token_usage": 0, "total_sentences": 0}
    work_dir = f"/content/{task_id}"
    os.makedirs(work_dir, exist_ok=True)

    local_pdf = os.path.join(work_dir, "input.pdf")
    local_mp3 = os.path.join(work_dir, "input.mp3")

    try:
        # 1. Download Files from Supabase
        print(f"[{task_id}] Downloading assets...")
        pdf_data = supabase.storage.from_(bucket_name).download(pdf_storage_path)
        with open(local_pdf, 'wb') as f: f.write(pdf_data)

        mp3_data = supabase.storage.from_(bucket_name).download(mp3_storage_path)
        with open(local_mp3, 'wb') as f: f.write(mp3_data)

        telemetry['pdf_size_bytes'] = os.path.getsize(local_pdf)

        # 2. Extract Text with Marker
        print(f"[{task_id}] Running Marker PDF...")
        marker_out = os.path.join(work_dir, "marker_out")
        os.makedirs(marker_out, exist_ok=True)
        subprocess.run(["marker_single", local_pdf, "--output_dir", marker_out], check=True)

        text_path = None
        for root, dirs, files in os.walk(marker_out):
            for file in files:
                if file.endswith(".md"):
                    text_path = os.path.join(root, file)
                    break
            if text_path: break

        if not text_path:
            raise RuntimeError("Marker failed to extract text from PDF.")

        with open(text_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()

        # 3. Groq AI Director
        print(f"[{task_id}] Requesting Groq AI Director...")
        client = Groq(api_key=os.getenv("GROQ_API_KEY"))
        system_prompt = """You are a strict text annotation tool. Break the text down into individual sentences. Prepend a descriptive emotion tag enclosed in parentheses at the VERY BEGINNING of every sentence. Output EACH tagged sentence on its own new line. Output ONLY the tagged text."""

        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Here is the raw text:\n{raw_text}"}
            ],
            temperature=0.3,
            max_tokens=8000
        )

        tagged_text = completion.choices[0].message.content.strip()
        tagged_text = tagged_text.replace("```markdown", "").replace("```", "").strip()
        telemetry["token_usage"] = completion.usage.total_tokens

        # 4. Prepare Audio
        print(f"[{task_id}] Trimming reference audio...")
        reference_audio = os.path.join(work_dir, "reference.wav")
        audio = AudioSegment.from_mp3(local_mp3)
        audio = audio[:10000]
        audio.export(reference_audio, format="wav")

        # 5. Load VoxCPM & Synthesize
        print(f"[{task_id}] Loading VoxCPM2 Model...")
        model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=True)

        sentences = [s.strip() for s in tagged_text.split('\n') if len(s.strip()) > 5]
        telemetry["total_sentences"] = len(sentences)
        all_audio = []

        for i, sentence in enumerate(sentences):
            print(f"[{task_id}] Synthesizing {i+1}/{len(sentences)}")
            wav = model.generate(
                text=sentence,
                reference_wav_path=reference_audio,
                cfg_value=2.5,
                inference_timesteps=10
            )
            all_audio.append(wav)

        if not all_audio:
            raise RuntimeError("Failed to generate any audio sentences.")

        final_wav = np.concatenate(all_audio, axis=0)
        output_file = os.path.join(work_dir, "final_audiobook.wav")
        sf.write(output_file, final_wav, model.tts_model.sample_rate)

        if torch.cuda.is_available():
            telemetry['peak_gpu_memory_gb'] = round(torch.cuda.max_memory_allocated() / (1024**3), 2)

        # 6. Upload Outputs to Supabase
        print(f"[{task_id}] Uploading artifacts to Supabase...")
        output_blob_path = f"outputs/{task_id}/audiobook.wav"

        # Upload audio
        with open(output_file, 'rb') as f:
            supabase.storage.from_(bucket_name).upload(
                path=output_blob_path,
                file=f.read(),
                file_options={"content-type": "audio/wav"}
            )

        # Upload analytics
        telemetry["total_processing_time_seconds"] = round(time.time() - start_time, 2)
        analytics_file = os.path.join(work_dir, "analytics.json")
        with open(analytics_file, 'w') as f:
            json.dump(telemetry, f)

        with open(analytics_file, 'rb') as f:
            supabase.storage.from_(bucket_name).upload(
                path=f"outputs/{task_id}/analytics.json",
                file=f.read(),
                file_options={"content-type": "application/json"}
            )

        # Generate Public URL
        public_url = supabase.storage.from_(bucket_name).get_public_url(output_blob_path)

        return {
            "status": "SUCCESS",
            "audiobook_url": public_url,
            "telemetry": telemetry
        }

    except Exception as e:
        print(f"[{task_id}] ERROR: {str(e)}")
        raise e

    finally:
        print(f"[{task_id}] Cleaning up local resources...")
        if os.path.exists(work_dir):
            shutil.rmtree(work_dir, ignore_errors=True)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

Writing worker.py


In [ ]:
!PYTHONPATH=/content celery -A worker worker --pool=prefork --concurrency=1 --max-tasks-per-child=1 --loglevel=info

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/celery/platforms.py:841: SecurityWarning: You're running the worker with superuser privileges: this is
absolutely not recommended!

Please specify a different user using the --uid option.

User information: uid=0 euid=0 gid=0 egid=0

  warnings.warn(SecurityWarning(ROOT_DISCOURAGED.format(
[2026-05-01 18:33:56